In [39]:
import matplotlib.pyplot as plt

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from src.BBOSSI import Replicator
from src.MonteCarlo import SimPEJ, SimJDM
from collections.abc import Iterable

# functions

In [3]:
def get_weights(x):
    weights = []
    try:
        d = len(x)
        w = 1
        for i in range(d):
            weights.append(w*np.cos(x[i])**2)
            w = w * np.sin(x[i])**2
        weights.append(w)
    except:
        w = np.cos(x)**2
        weights = [w, 1-w]
    return np.array(weights)

def get_DW(x):
    try:
        d = len(x)
        w = get_weights(x)
        dw = np.zeros(shape=(d,d+1))
        for i in range(d):
            dw[i,i] = -2 * np.tan(x[i])
            dw[i,i+1:] = 2 / np.tan(x[i])
        dw = w * dw
    except:
        dw = np.sin(x) * np.cos(x) * np.array([-2, 2])
    return dw
    
def get_Gradient(x, Q):
    dW = get_DW(x)
    dMu = 2 * Q @ get_weights(x)
    return dW @ dMu

# Problem Description




## Stocks prices 

**Black-Scholes model** characterizes the path of assets prices $S_t$.
$$
\begin{equation*}
\frac{dS(t)}{S(t)} = \mu dt + \sigma dW(t).
\end{equation*}
$$

**Jump-diffusion model** 
$$
\begin{equation*}
\frac{dS(t)}{S(t)} = \mu dt + \sigma dW(t) + dJ(t),
\end{equation*}
$$
where $W(t)$ is a standard Wiener porcess, $J(t) = \sum_{j=1}^{N(t)} Z_j$ is a coumpund Poisson process: $N(t)$ is a Poisson process with arrival rate $\lambda$ and $Z_j,j=1,2,\cdots$ are identical and independently distributed.  Assume that $W(t)$, $N(t)$ and $Z_j$ are mutually independent.

**Market influence**
Suppose the investments can influence some stocks, of which the set of the indexes are denoted by $\mathcal{I}$, via the jumps, i.e.,
$$
\begin{equation*}
J_k(\Delta) = \sum_{j=1}^{N(\Delta)} Z_{j,k}, \quad k \in [K],
\end{equation*}
$$
where $N(\Delta) \sim {\sf Poisson}(\lambda\Delta)$, $Z_{j,k} \stackrel{i.i.d.}{\sim} {\sf Exp}(1/w_k)$ if $k \in \mathcal{I}$, otherwise $Z_{j,k} = 0~ w.p.1$.

## Selection criteria
**Markowitz model** is introduced by *Harry Markowitz* (see [Markowitz, H. Portfolio selection. *Journal of Finance* (1952)](https://doi.org/10.1111/j.1540-6261.1952.tb01525.x), and recent reviews [Rubinstein 2002](https://doi.org/10.1111/1540-6261.00453), [Zhang et al 2018](https://doi.org/10.1007/s10700-017-9266-z)). A fundamental formulation is given by
$$
\begin{align*}
\min_w ~& w^{\sf T} \Sigma w \\
s.t.   ~& \tilde\mu^{\sf T}w \geq R, \\
        & e^{\sf T}w = 1, \\
        & w \geq 0. \\
\end{align*}
$$
For each $R$, there exists some $\rho \geq 0$ such that \eqref{model.markowitz} is equivalent to 
$$
\begin{equation} \begin{split}
\min_{w\geq 0} ~& w^{\sf T} \Sigma w - \rho \tilde\mu^{\sf T} w \\
s.t. ~~         & e^{\sf T} w =1.
\end{split} \end{equation}
$$
Following the models, we can express the above paramters by
$$
\begin{equation*}
\begin{split}
    \tilde\mu &= \mu + \lambda \Delta \sum_{i\in \mathcal{I}} e_i w_i, \\
    \Sigma &= \sigma \sigma^{\sf T} + 2\lambda \Delta {\sf diag}\left(\sum_{i\in \mathcal{I}} e_i w_i^2\right).
\end{split}
\end{equation*}
$$

**Markowitz model** can be treat as a stochastic optimizaiton problem when $mu$ is known.
$$
\begin{align*}
\min_{w\geq 0} ~& w^{\sf T} \mathbb{E}\left[X X^{\sf T}\right] w 
s.t. ~~         & e^{\sf T} w = 1
\end{align*}
$$

In [35]:
class Portfolio(object):
    checkParam = True
    weight = .5 + np.zeros(2)
    def __init__(self, mu=np.zeros(2), Sigma=np.eye(2), w=[], jump:list=[False]):
        self.__mu = np.array(mu)
        self.__sigma = np.linalg.cholesky(Sigma)
        self.checkParam = len(self.__mu)!=self.__sigma.shape[0] or len(self.__mu)!=self.__sigma.shape[1]
        if self.checkParam:
            print("!!!?")
            self.__mu = np.zeros(2)
            self.__sigma = np.eye(2)
            print("Assign values by default...")
        self.__dim = len(self.__mu)
        self.__jump = [False] * self.__dim

        if len(w) == self.__dim:
            self.weight = np.array(w)
        else:
            if self.__dim != 2:
                self.weight = np.array([1/self.__dim] * self.__dim)
            print("Assign weights by default...")
        self.__jump = jump
        self.rvJump = SimPEJ()

    def checkInit(self):
        print("mu = {}".format(self.__mu))
        print("sigma = {}".format(self.__sigma))
        print("dim = {}".format(self.__dim))
        print("jump = {}".format(self.__jump))
        print("weights = {}".format(self.weight))


    def simulate(self, size:int=1, t:float=1):
        t = abs(t)
        rvBase = np.random.normal(size=(self.__dim, size))
        rvBase = t * self.__mu.T + np.sqrt(t) * (self.__sigma @ rvBase).T
        rvJump = np.zeros(shape=(size, self.__dim))
        if self.__jump:
            sclr = np.zeros(self.__dim)
            sclr[self.__jump] = sclr[self.__jump] + 1/self.weight[self.__jump].copy()
            rvJump = self.rvJump.rvs(t, self.__dim, sclr, size)
        return rvBase + rvJump

In [55]:
def fn1(theta, portfolio, tau=1):
    portfolio.weight = get_weights(theta)
    rvs = portfolio.simulate(tau) @ portfolio.weight
    return np.mean(rvs)

In [36]:
mu = np.zeros(3)
Sigma = np.array([
    [1, -2, -3],
    [-2, 9, 0],
    [-3, 0, 25]
])
jump = [False, False, True]
test = Portfolio(mu, Sigma, jump=jump)
test.checkInit()

Assign weights by default...
mu = [0. 0. 0.]
sigma = [[ 1.          0.          0.        ]
 [-2.          2.23606798  0.        ]
 [-3.         -2.68328157  2.96647939]]
dim = 3
jump = [False, False, True]
weights = [0.33333333 0.33333333 0.33333333]


In [44]:
rv = test.simulate(1000)
np.cov(rv.T)

array([[ 0.95406285, -1.96068941, -2.67739316],
       [-1.96068941,  8.81656644, -0.34505029],
       [-2.67739316, -0.34505029, 69.31114978]])

In [6]:
jump = np.zeros(shape=(100,3))
s = - 5 * np.log(np.random.rand(100))
add = (s <= 1)

In [9]:
rvBase = -np.log(np.random.uniform(size=(100,3)))

In [4]:
rvJump = jump.simulate(1, 1, 100)
print(rvJump.shape)

(100, 1)


# Numerical Settings
We introduce a new parameter vector $\theta = \left( \vartheta_1, \vartheta_2, \cdots, \vartheta_{K-1}\right)^{\sf T}$ with $\vartheta_k \in [0, \pi/2]$, $k=1,\cdots,K-1$ and express the weights in terms of $\vartheta_k$'s as follows:
$$\begin{equation*}
w_k(\theta) = \begin{cases}
    \cos^2(\vartheta_1), & k=1; \\
    \cos^2(\vartheta_k) \prod_{i=1}^{k-1} \sin^2(\vartheta_i), & 2\leq k \leq K-1; \\
    \prod_{i=1}^{k-1} \sin^2(\vartheta_i), & k = K.
\end{cases}
\end{equation*}
$$
In cases I and II, $\lambda$, $\mathcal{I}$ and $\sigma$ are unknown to the investor.

## Case I
Set $K = 3$, $\lambda = 2$, $\Delta = \rho = 1$, $\mu = {\bf 0}$, $\mathcal{I} = \{3\}$ and
$$
\begin{equation*}
\sigma = \left(\begin{matrix}
    \sqrt{3}/4 & 1/2 & 3/4 \\
    2 & \sqrt{5} & 0 \\
    3 & 0 & 4
\end{matrix}\right).
\end{equation*}
$$

In [38]:
Sigma = np.array([
    [1, -2, -3],
    [-2, 9, 0],
    [-3, 0, 25]
])
sigma = np.linalg.cholesky(Sigma)
mu = np.array([2,3,3])

In [56]:
rvs0 = np.random.normal(size=(3,1000))
rvs1 = ((sigma @ rvs0).T + mu).T
np.cov(rvs1)

array([[ 0.98257155, -2.03408302, -2.82425344],
       [-2.03408302,  9.28573068, -0.23564154],
       [-2.82425344, -0.23564154, 24.36560939]])

In [57]:
np.cov(rvs0)

array([[ 9.82571547e-01, -3.08308721e-02,  1.37311904e-02],
       [-3.08308721e-02,  1.01593696e+00,  7.01550565e-04],
       [ 1.37311904e-02,  7.01550565e-04,  1.01814263e+00]])

## Case II
Introduce a riskless assets with $\mu_0=1$, and reset $\mu = \left(2, 3, 3\right)^{\sf T}$.

## Case III: target strategy


In [5]:
get_Gradient(1, np.eye(2))

0.7568024953079279

In [7]:
Q = np.array([
    [4, -2, -4],
    [-2, 5, 0],
    [-4, 0, 6]
])

In [8]:
x = [1,1]
w = get_weights(x)
print(w)
dMu = 2 * Q @ w
print(dMu)
dW = get_DW(x)
print(dW)

[0.29192658 0.20670545 0.50136797]
[-2.50235288  0.8993482   3.68100293]
[[-0.90929743  0.26544809  0.64384934]
 [ 0.         -0.64384934  0.64384934]]
